# Deteksi Anomali Banjir dengan Isolation Forest per Kota

Notebook ini memakai dataset yang sama (`processed/dataset_timeseries.csv`), tetapi implementasinya difokuskan ke **unsupervised anomaly detection**. Model utama adalah **Isolation Forest**, bukan supervised classifier.

Perubahan metodologi utama:
- model dilatih terpisah untuk **Balikpapan** dan **Samarinda**;
- setiap kota punya train/test split stratified sendiri;
- `StandardScaler` dan `IsolationForest` hanya di-fit pada train fold kota tersebut;
- test fold hanya di-transform dan diberi `anomaly_score` serta `is_anomaly`;
- `is_anomaly = 1` diinterpretasikan sebagai **indikasi banjir**;
- tidak ada FLAML, Random Forest, XGBoost, CatBoost, SMOTE, confusion matrix, atau classification report.


## 0. Clone Repository (Google Colab)
Saat berjalan di Colab, isi `REPO_URL` dengan repository yang berisi dataset dan folder `scripts/`.

In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/Noelsip/flood-bpn-smd.git"

def in_colab():
    return "google.colab" in sys.modules

if in_colab():
    repo_dir = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(repo_dir)
    print("Direktori kerja:", os.getcwd())
else:
    print("Bukan Colab: memakai repository lokal.")


## 1. Dependensi
Hanya dependensi inti yang dibutuhkan untuk deteksi anomali tabular.

In [ ]:
if in_colab():
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pandas", "scikit-learn", "matplotlib"
    ], check=True)
    print("Dependensi terpasang.")
else:
    print("Bukan Colab: pastikan requirements.txt sudah terpasang.")


## 2. Import Library dan Konfigurasi

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.isolation_forest_per_city import (
    DEFAULT_CITIES,
    TARGET,
    attach_kecamatan_prior,
    feature_columns,
    run_per_city_isolation_forest,
    save_outputs,
)

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

SEED = 42
ANOMALY_CONTAMINATION = 0.05
TRAIN_RATIO = 0.80
SPLIT_MODE = "stratified_label_per_city"
MAX_KECAMATAN_PER_ALERT = 3

ROOT = Path.cwd()
DATA_PATH = ROOT / "processed" / "dataset_timeseries.csv"
OUT_DIR = ROOT / "outputs"
CLEAN_DIR = ROOT / "clean"
OUT_DIR.mkdir(exist_ok=True)

print("Dataset:", DATA_PATH)
print("Output :", OUT_DIR)


## 3. Pemuatan Dataset
Dataset tetap sama. Data kota luar tidak dipakai untuk melatih model utama karena revisi memilih model yang belajar pola normal **per kota**. Split dilakukan secara stratified di dalam tiap kota: 80% data banjir dan 80% data tidak banjir masuk train, sisanya menjadi test.

In [ ]:
data = pd.read_csv(DATA_PATH, parse_dates=["time"])
city_data = data[data["city"].isin(DEFAULT_CITIES)].copy()
features = feature_columns(city_data)

print("Seluruh dataset:", data.shape)
print("Dataset Kaltim:", city_data.shape)
print("Jumlah fitur numerik:", len(features))
print("Split mode:", SPLIT_MODE, "| train ratio:", TRAIN_RATIO)
print("Kota:", city_data["city"].unique().tolist())

if "split" in city_data.columns:
    display(city_data.groupby(["split", "city"])[TARGET].agg(hari="count", banjir_historis="sum"))
else:
    display(city_data.groupby("city")[TARGET].agg(hari="count", banjir_historis="sum"))


## 4. Eksplorasi Singkat
Bagian ini hanya deskriptif. Label historis banjir boleh ditampilkan sebagai konteks data, tetapi tidak dipakai untuk melatih Isolation Forest.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
city_data.groupby("city").size().plot.bar(ax=ax[0], color="#2A6F97", rot=0)
ax[0].set_title("Jumlah observasi per kota")
ax[0].set_xlabel("")

if TARGET in city_data.columns:
    city_data.groupby("city")[TARGET].sum().plot.bar(ax=ax[1], color="#D1495B", rot=0)
    ax[1].set_title("Hari banjir historis per kota")
    ax[1].set_xlabel("")
else:
    ax[1].axis("off")

plt.tight_layout()
plt.show()


## 5. Isolation Forest per Kota
Setiap kota diproses sendiri. Di dalam tiap kota, data banjir dan tidak banjir historis dipisah lebih dulu untuk membentuk train 80% dan test 20% secara stratified. Label historis hanya dipakai untuk pembagian data, sedangkan scaler dan Isolation Forest hanya fit pada fitur train fold kota itu.

In [ ]:
summary_df, anomaly_df, city_results = run_per_city_isolation_forest(
    city_data,
    cities=DEFAULT_CITIES,
    contamination=ANOMALY_CONTAMINATION,
    train_ratio=TRAIN_RATIO,
    split_mode=SPLIT_MODE,
    random_state=SEED,
)

kecamatan_df = attach_kecamatan_prior(
    anomaly_df,
    CLEAN_DIR,
    max_per_city=MAX_KECAMATAN_PER_ALERT,
    anomalies_only=True,
)
save_outputs(summary_df, anomaly_df, OUT_DIR, kecamatan_df)

summary_view = summary_df.copy()
for col in ["train_start", "train_end", "test_start", "test_end"]:
    summary_view[col] = pd.to_datetime(summary_view[col]).dt.date

display(summary_view)
print("Output disimpan:")
print("-", OUT_DIR / "isolation_forest_summary_per_city.csv")
print("-", OUT_DIR / "isolation_forest_anomaly_all.csv")
print("-", OUT_DIR / "isolation_forest_anomaly_balikpapan.csv")
print("-", OUT_DIR / "isolation_forest_anomaly_samarinda.csv")
print("-", OUT_DIR / "isolation_forest_anomaly_kecamatan.csv")


## 6. Ringkasan Anomali Test per Kota
Evaluasi dipisah per kota dan tidak memakai confusion matrix. Kolom banjir historis hanya konteks untuk melihat irisan tanggal, bukan target supervised.

In [ ]:
cols = [
    "city", "test_days", "detected_anomaly", "anomaly_rate",
    "mean_anomaly_score", "max_anomaly_score",
    "historical_flood_days", "anomaly_on_historical_flood",
]
display(summary_df[cols].round(4))

for city, result in city_results.items():
    top = result.test.sort_values("anomaly_score", ascending=False).head(10)
    show_cols = [
        col for col in [
            "city", "time", "anomaly_score", "is_anomaly", "flood_anomaly", "anomaly_interpretation",
            TARGET, "rain", "precip", "soil", "rain_roll7_sum", "precip_roll7_sum", "soil_roll7_mean",
        ] if col in top.columns
    ]
    print(f"Top anomali test - {city}")
    display(top[show_cols])


## 7. Visualisasi Skor Anomali
Grafik memperlihatkan distribusi skor pada test fold masing-masing kota dan garis ambang empiris yang memisahkan data normal/anomali menurut model.

In [ ]:
fig, axes = plt.subplots(1, len(city_results), figsize=(13, 4), squeeze=False)
for ax, (city, result) in zip(axes.ravel(), city_results.items()):
    test = result.test.copy()
    normal = test[test["is_anomaly"] == 0]
    anomaly = test[test["is_anomaly"] == 1]
    ax.hist(normal["anomaly_score"], bins=30, alpha=0.75, label="normal", color="#4C9F70")
    ax.hist(anomaly["anomaly_score"], bins=30, alpha=0.75, label="indikasi banjir", color="#D1495B")
    if not anomaly.empty:
        ax.axvline(anomaly["anomaly_score"].min(), color="#222222", linestyle="--", linewidth=1)
    ax.set_title(city)
    ax.set_xlabel("anomaly_score")
    ax.set_ylabel("jumlah hari")
    ax.legend()
plt.tight_layout()
plt.show()


## 8. Output Akhir
`is_anomaly = 1` adalah hasil deteksi anomali cuaca yang diinterpretasikan sebagai indikasi banjir. Kecamatan hanya dilampirkan pada output anomali sebagai prior lokasi historis dari `clean/banjir_*.csv`; kolom ini tidak digunakan untuk training model.

In [ ]:
output_cols = [
    col for col in [
        "city", "time", "anomaly_score", "is_anomaly", "flood_anomaly", "anomaly_interpretation",
        TARGET, "rain", "precip", "soil", "rain_roll3_sum", "rain_roll7_sum", "rain_roll14_sum",
        "precip_roll3_sum", "precip_roll7_sum", "soil_roll3_mean", "soil_roll7_mean",
    ] if col in anomaly_df.columns
]
final_output = anomaly_df[output_cols].sort_values(["city", "time"]).reset_index(drop=True)
final_kecamatan_cols = [
    col for col in [
        "city", "time", "kecamatan", "banjir_historis", "anomaly_score", "is_anomaly",
        "flood_anomaly", "anomaly_interpretation", TARGET, "rain", "precip", "soil",
        "rain_roll7_sum", "precip_roll7_sum", "soil_roll7_mean",
    ] if col in kecamatan_df.columns
]
final_kecamatan = kecamatan_df[final_kecamatan_cols].sort_values(
    ["city", "time", "banjir_historis"], ascending=[True, True, False]
).reset_index(drop=True)

display(final_output.head(20))
print("Output anomali + prior kecamatan:")
display(final_kecamatan.head(30))

final_output.to_csv(OUT_DIR / "isolation_forest_final_output.csv", index=False)
final_kecamatan.to_csv(OUT_DIR / "isolation_forest_final_output_kecamatan.csv", index=False)
print("Disimpan:", OUT_DIR / "isolation_forest_final_output.csv")
print("Disimpan:", OUT_DIR / "isolation_forest_final_output_kecamatan.csv")


---
## Catatan Metodologi
- Pendekatan ini adalah **unsupervised anomaly detection**, bukan supervised flood classification.
- Model dilatih **terpisah per kota** agar pola normal Balikpapan dan Samarinda tidak dicampur.
- Untuk setiap kota, data banjir dan tidak banjir dibagi 80/20 lebih dulu, lalu `StandardScaler.fit()` dan `IsolationForest.fit()` hanya memakai train fold kota tersebut.
- Test fold hanya dipakai untuk menghasilkan skor dan label anomali.
- Kecamatan pada output adalah prior lokasi historis per kota, bukan hasil training atau fitur model.
- Tidak ada confusion matrix karena output utama bukan prediksi supervised terhadap kelas banjir, melainkan deteksi outlier/anomali cuaca sebagai indikasi banjir.
